**LIBRARY IMPORTS AND PARAMETER DEFINITIONS**

In [ ]:
import os
from tqdm.notebook import trange
import numpy as np
import time
import secrets

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
CRYPTO_TARGET = 'TINYAES128C'
SS_VER = 'SS_VER_1_1'

# Firmware implementations
unmasked_asm = "keccak_unmasked_Opt_ASSEMBLY"  
masked_ti_1st_order_asm = "keccak_masked_TI3shares_1stOrder_Opt_ASSEMBLY"   # 1st-order TI, 3 shares
masked_ti_2nd_order_asm = "keccak_masked_TI3shares_2ndOrder_Opt_ASSEMBLY"   # 1st+2nd-order TI, 3 shares

base_dir_hex = ".\\..\\..\\firmware\\mcu"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% DEFINE PARAMETERS HERE %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
base_dir_traces = "F:"   # Base directory where traces will be stored
experiment_name = "DATASET_KECCAK_TIDOM"                   # Dataset subfolder name

masking_list = [masked_ti_1st_order_asm]                   # List of firmware implementations to evaluate
remasking_list = ["no-remasking"]

iteration_start = 0                                          # Index of the first trace batch
iteration_end = 20000000000                                  # Index of the last trace batch

iterations_per_full_trace = 11                               # Number of concatenated sub-traces per full trace (3 for unmasked, 11 for masked)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


**INITIALIZATION OF CHIPWHISPERER PLATFORM**

In [ ]:
# CHIPWHISPERER-LITE CW308 PLATFORM INITIALIZATION
%run "../Setup_Scripts/Setup_Generic.ipynb"

**CODES COMPILATION**

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"

cd ../../firmware/mcu/keccak_unmasked_Opt_ASSEMBLY
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

cd ../keccak_masked_TI3shares_1stOrder_Opt_ASSEMBLY/no-remasking
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

cd ../../keccak_masked_TI3shares_2ndOrder_Opt_ASSEMBLY/no-remasking
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

**DEFINITION OF FUNCTIONS FOR TRACES ACQUISITIONS**

In [ ]:
import os
import h5py
import numpy as np
from tqdm import trange
import secrets

# ================== CONFIGURATION ==================
TRACES_PER_FILE = 5000
SAMPLES_PER_ITERATION = 24400

# ================== HDF5 HANDLING ==================
def open_hdf5_file(file_index, experiment_name, base_dir_traces,
                   total_samples_per_trace):

    file_path = os.path.join(
        base_dir_traces,
        experiment_name,
        f"{experiment_name}_traces_part{file_index:03d}.h5"
    )

    f = h5py.File(file_path, 'w')  # Always create a new file

    dset_traces = f.create_dataset(
        'traces',
        shape=(TRACES_PER_FILE, total_samples_per_trace),
        maxshape=(TRACES_PER_FILE, total_samples_per_trace),
        dtype='float32'
    )

    dset_plaintexts = f.create_dataset(
        'plaintexts',
        shape=(TRACES_PER_FILE, 16),
        dtype='uint8'
    )

    dset_masks = f.create_dataset(
        'masks',
        shape=(TRACES_PER_FILE, 16),
        dtype='uint8'
    )

    dset_seeds = f.create_dataset(
        'seeds',
        shape=(TRACES_PER_FILE, 32),
        dtype='uint8'
    )

    return f, dset_traces, dset_plaintexts, dset_masks, dset_seeds


def capture_traces(use_seed, experiment_name):
    scope.gain.db = 0.1

    scope.adc.samples = 24400
    scope.adc.timeout = 2

    ktp = cw.ktp.Basic()
    key, text = ktp.next()

    fixed_plaintext = bytes([108, 81, 56, 195, 111, 10, 64, 204,
                             0, 43, 197, 36, 103, 202, 212, 146])

    target.baud = 38400

    os.makedirs(os.path.join(base_dir_traces, experiment_name), exist_ok=True)

    total_samples = iterations_per_full_trace * SAMPLES_PER_ITERATION

    # ---- File state ----
    file_index = 0
    trace_index = 0

    f, dset_traces, dset_plaintexts, dset_masks, dset_seeds = open_hdf5_file(
        file_index, experiment_name, base_dir_traces, total_samples
    )

    try:
        for _ in range(iteration_start, iteration_end):
            full_trace = np.empty(total_samples, dtype=np.float32)

            for i in trange(10, desc='Capturing traces'):
                scope.arm()
                target.flush()

                # -------- PLAINTEXT SELECTION --------
                # Fixed vs. Random
                if i % 2:
                    key, text = ktp.next()
                else:
                    text = fixed_plaintext

                if use_seed:
                    # -------- CHACHA20 SEED GENERATION --------
                    external_seed = [int.from_bytes(secrets.token_bytes(1), 'little') for _ in range(32)]

                    for k in range(2):
                        target.simpleserial_write('s', external_seed[k * 16:(k + 1) * 16])
                        _ = target.simpleserial_read('r', 16)

                    # -------- PLAINTEXT INITIAL MASKING --------
                    mask = [int.from_bytes(secrets.token_bytes(1), 'little') for _ in range(16)]
                    masked_text = [m ^ t for m, t in zip(mask, text)]

                    # -------- TRANSMIT MASKED PLAINTEXT --------
                    target.simpleserial_write('k', mask)
                    _ = target.simpleserial_read('r', 16)

                    for k in range(iterations_per_full_trace):
                        scope.adc.offset = k * 24400
                        trace = cw.capture_trace(scope, target, masked_text)
                        start = k * SAMPLES_PER_ITERATION
                        end = start + SAMPLES_PER_ITERATION
                        full_trace[start:end] = trace.wave
                else:
                    # -------- UNMASKED PLAINTEXT TRANSMISSION --------
                    for k in range(iterations_per_full_trace):
                        scope.adc.offset = k * 24400
                        trace = cw.capture_trace(scope, target, text)
                        start = k * SAMPLES_PER_ITERATION
                        end = start + SAMPLES_PER_ITERATION
                        full_trace[start:end] = trace.wave

                # -------- DATA STORAGE --------
                dset_traces[trace_index, :] = full_trace
                dset_plaintexts[trace_index, :] = np.frombuffer(text, dtype=np.uint8)

                if use_seed:
                    dset_masks[trace_index, :] = mask
                    dset_seeds[trace_index, :] = external_seed

                trace_index += 1

                # -------- FILE ROTATION --------
                if trace_index >= TRACES_PER_FILE:
                    f.close()
                    file_index += 1
                    trace_index = 0
                    f, dset_traces, dset_plaintexts, dset_masks, dset_seeds = open_hdf5_file(
                        file_index, experiment_name, base_dir_traces, total_samples
                    )
    finally:
        f.close()


**MICROCONTROLLER FLASHING AND INITIALIZATION OF TRACES ACQUISITION**

In [ ]:
# FLASH STM32F3 TARGET AND RUN TRACE ACQUISITION
for masking in masking_list:
    masking_path = os.path.normpath(os.path.join(base_dir_hex, masking))

    if masking in ["keccak_unmasked_NoOpt_ASSEMBLY", "keccak_unmasked_Opt_ASSEMBLY"]:
        for filename in os.listdir(masking_path):
            if filename.endswith(".hex"):
                hex_path = os.path.normpath(os.path.join(masking_path, filename))
                cw.program_target(scope, prog, hex_path.format(PLATFORM))
                capture_traces(use_seed=False, experiment_name="exp_" + masking)
    else:
        for remasking in remasking_list:
            remasking_path = os.path.normpath(os.path.join(base_dir_hex, masking, remasking))
            for filename in os.listdir(remasking_path):
                if filename.endswith(".hex"):
                    hex_path = os.path.normpath(os.path.join(remasking_path, filename))
                    cw.program_target(scope, prog, hex_path.format(PLATFORM))
                    capture_traces(
                        use_seed=True,
                        experiment_name="exp_" + masking + "_" + remasking
                    )
